In [1]:
import mlflow
import os
from dotenv import load_dotenv

from mlflow.genai import scorer
from datasets import load_dataset


load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "log_model_with_prompt"

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("ag_news", split="train")
df = dataset.to_pandas()

df["label"] = df["label"].map({0: "World", 1: "Sports", 2: "Business", 3: "Science"})

df = df.sample(frac=1).reset_index(drop=True)
df.head()

,text,label
0,First US commercial flight in 30 years lands i...,Business
1,Pro-Pakistan rebels claim Kashmir attack A pro...,World
2,"Bush, Kerry focus on Florida as polls indicate...",World
3,Rank Group mulls break-up Shares in Rank Group...,Business
4,Sorry Anelka still on the trading block Anelka...,Sports


In [3]:
prompt_uri = "prompts:/news_classifier/1"

NUM_SAMPLES = 3
train_data = []
for i in range(NUM_SAMPLES):
    article = df.iloc[i]["text"]
    expected = df.iloc[i]["label"]
    eval_dict = {
        "inputs": {"article": article, "prompt_uri": prompt_uri},
        "expectations": {"expected_response": expected},
    }
    train_data.append(eval_dict)

train_data[0]

{'inputs': {'article': 'First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.',
  'prompt_uri': 'prompts:/news_classifier/1'},
 'expectations': {'expected_response': 'Business'}}

In [4]:
# Create a detailed prompt for classification
prompt = """
You are a helpful assistant that can classify news articles into one of the following categories:
- World
- Sports
- Business
- Science
Article: {article}
"""

initial_prompt = mlflow.genai.register_prompt(
    name="news_classifier",
    template=prompt,
)

2025/10/27 18:11:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 1


In [5]:
with mlflow.start_run(run_name="langchain_model"):
    model_info = mlflow.pyfunc.log_model(
        name="news_classifier_with_prompt",
        python_model="lc_model_with_prompt.py",
        prompts=[prompt_uri],
        input_example=[train_data[0]["inputs"]],
    )

2025/10/27 18:11:51 INFO mlflow.models.signature: Running the predict function to generate output based on input example


Loading context
Received Input:  {'article': 'First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.', 'prompt_uri': 'prompts:/news_classifier/1'}


2025/10/27 18:12:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Loading context
Received Input:  {'article': 'First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.', 'prompt_uri': 'prompts:/news_classifier/1'}


2025/10/27 18:12:37 INFO mlflow.models.model: Found the following environment variables used during model inference: [GOOGLE_API_KEY, OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run langchain_model at: http://localhost:5000/#/experiments/581752715953272221/runs/61d5246cce1a46adabcf05ae4c3a9936
🧪 View experiment at: http://localhost:5000/#/experiments/581752715953272221


In [6]:
model = mlflow.pyfunc.load_model(model_uri=model_info.model_uri)

Loading context


In [7]:
model.predict([train_data[0]["inputs"]])

Received Input:  {'article': 'First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.', 'prompt_uri': 'prompts:/news_classifier/1'}


['World']

In [9]:
# model.predict(["str"])

In [10]:
def predict_fn(article, prompt_uri):
    print("Article: ", article)
    response = model.predict([{"article": article, "prompt_uri": prompt_uri}])
    return response

In [11]:
predict_fn(train_data[0]["inputs"]["article"], train_data[0]["inputs"]["prompt_uri"])

Article:  First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.
Received Input:  {'article': 'First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.', 'prompt_uri': 'prompts:/news_classifier/1'}


['World']

In [13]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations["expected_response"]
    return outputs[0] == expectations


with mlflow.start_run(run_name="evaluation"):
    results = mlflow.genai.evaluate(
        data=train_data,
        scorers=[exact_match],
        predict_fn=predict_fn,
        model_id=model.model_id,
    )

2025/10/27 18:16:32 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Article:  First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.
Received Input:  {'article': 'First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.', 'prompt_uri': 'prompts:/news_classifier/1'}


2025/10/27 18:16:33 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-1c3860f170344a54b200dc7c64fd231f
2025/10/27 18:16:33 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


Article:  First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.
Received Input:  {'article': 'First US commercial flight in 30 years lands in Vietnam HO CHI MINH CITY, VIETNAM - The first US passenger jet to fly to Vietnam since the war ended almost 30 years ago landed in Ho Chi Minh City Friday.', 'prompt_uri': 'prompts:/news_classifier/1'}
Article:  Pro-Pakistan rebels claim Kashmir attack A pro-Pakistan rebel group has claimed responsibility for a landmine attack that has killed 11 people in Kashmir. The Current News Service reports that Hizbul Mujahedin also claims to have snatched six AK 
Received Input:  {'article': 'Pro-Pakistan rebels claim Kashmir attack A pro-Pakistan rebel group has claimed responsibility for a landmine attack that has killed 11 people in Kashmir. The Current News Service reports that Hizbul Mujahedin also clai

Evaluating: 100%|██████████| 3/3 [Elapsed: 00:04, Remaining: 00:00] 
